In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score

import numpy as np

In [ ]:
df = pd.read_csv('/kaggle/input/-spotify-tracks-dataset/dataset.csv')
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)
df.dropna(inplace=True)
df_cleaned = df.drop_duplicates(subset=['track_name'], keep='first').copy()

In [ ]:
numerical_features = ['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
categorical_features = ['artists', 'track_genre']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop'
)
all_features_transformed = preprocessor.fit_transform(df_cleaned)

In [ ]:

name_to_index = {name: i for i, name in enumerate(df_cleaned['track_name'])}
index_to_name = {i: name for i, name in enumerate(df_cleaned['track_name'])}

In [ ]:
sse = []  
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(all_features_transformed)
    sse.append(kmeans.inertia_)  

plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), sse, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('SSE')  
plt.title('Elbow Method for Optimal K (SSE)')
plt.show()


optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
df_cleaned['cluster_label'] = kmeans.fit_predict(all_features_transformed)

In [ ]:
silhouette_scores = []

for k in range(2, 11):  # Silhouette score is undefined for k=1
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(all_features_transformed)
    score = silhouette_score(all_features_transformed, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 6))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='green')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Optimal K')
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

# Loop through different K values
for k in range(2, 11):  # silhouette score requires at least 2 clusters
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(all_features_transformed)
    score = silhouette_score(all_features_transformed, cluster_labels)
    silhouette_scores.append(score)
    print(f"K={k}, Silhouette Score={score:.4f}")

# Plot the silhouette scores
plt.figure(figsize=(8, 6))
plt.plot(range(2, 11), silhouette_scores, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Method for Optimal K")
plt.show()

In [ ]:
df_cleaned = df_cleaned.reset_index(drop=True)

def get_recommendations_clustered(song_name, k=10):
    if song_name not in name_to_index:
        print(f"Song '{song_name}' not found in the dataset.")
        return None
    song_index = name_to_index[song_name]
    song_cluster = df_cleaned.loc[song_index, 'cluster_label']
    cluster_indices = df_cleaned[df_cleaned['cluster_label'] == song_cluster].index.tolist()
    
    if hasattr(all_features_transformed, "toarray"):  # sparse matrix
        all_features_dense = all_features_transformed.toarray()
    else:  
        all_features_dense = all_features_transformed
    #Feature vector of all songs in the cluster
    cluster_features = all_features_dense[cluster_indices]
    #Feature vector of the inputted song
    song_features = all_features_dense[song_index].reshape(1, -1)
    
    # a KNN only on that cluster
    knn_cluster = NearestNeighbors(metric='cosine', algorithm='brute')
    knn_cluster.fit(cluster_features)
    
    # Finding neighbors
    distances, indices = knn_cluster.kneighbors(song_features, n_neighbors=k + 5)
    
    recommended_songs_list = []
    unique_songs = set()
    
    for idx in indices.flatten()[1:]:  
        cluster_song_index = cluster_indices[idx]
        track_name = df_cleaned.loc[cluster_song_index, 'track_name']
        artist_name = df_cleaned.loc[cluster_song_index, 'artists']
        
        if track_name not in unique_songs:
            recommended_songs_list.append([track_name, artist_name])
            unique_songs.add(track_name)
        
        if len(unique_songs) >= k:
            break
    
    return recommended_songs_list
    
song = input("Enter song name: ")
recommended_songs = get_recommendations_clustered(song)

if recommended_songs:
    print(f"\nRecommended songs for '{song}' :")
    for i, (track_name, artist_name) in enumerate(recommended_songs):
        print(f"{i+1}. {track_name} by {artist_name}")